# PCA y validacion cruzada (solucion)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_wine


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"


def is_colab():
    return "google.colab" in sys.modules


def ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if is_colab() and not target.exists():
        print(f"Clonando repositorio en {target} ...")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)


def resolve_data_dir():
    ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


In [ ]:
def load_dataset(data_dir):
    if data_dir is not None:
        base = data_dir / 'UCI-HAR-Dataset' / 'train'
        x_path = base / 'X_train.txt'
        y_path = base / 'y_train.txt'
        if x_path.exists() and y_path.exists():
            X = pd.read_csv(x_path, delim_whitespace=True, header=None)
            y = pd.read_csv(y_path, delim_whitespace=True, header=None).iloc[:, 0].astype(int) - 1
            return X, y, 'UCI-HAR train'

    wine = load_wine(as_frame=True)
    X = wine.data.copy()
    y = wine.target.astype(int).copy()
    return X, y, 'sklearn_wine'


DATA_DIR = resolve_data_dir()
X, y, fuente = load_dataset(DATA_DIR)
print('Fuente:', fuente)
print('Shape:', X.shape)


In [ ]:
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

pca = PCA()
pca.fit(X_s)

cum_var = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.searchsorted(cum_var, 0.95) + 1)
print('Componentes para 95% varianza:', n95)

plt.figure(figsize=(7.4, 4.1))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var, marker='o')
plt.axhline(0.95, color='r', ls='--')
plt.xlabel('Numero de componentes')
plt.ylabel('Varianza acumulada')
plt.title('PCA: varianza explicada acumulada')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)

pipe_no = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, multi_class='multinomial')),
])

pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n95)),
    ('clf', LogisticRegression(max_iter=5000, multi_class='multinomial')),
])

score_no = cross_val_score(pipe_no, X, y, cv=cv, scoring='accuracy')
score_pca = cross_val_score(pipe_pca, X, y, cv=cv, scoring='accuracy')

print(f'Sin PCA: media={score_no.mean():.4f}, desvio={score_no.std():.4f}')
print(f'Con PCA: media={score_pca.mean():.4f}, desvio={score_pca.std():.4f}')


In [ ]:
n_vals = range(2, min(X.shape[1], 25) + 1)
acc_vals = []
for n in n_vals:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n)),
        ('clf', LogisticRegression(max_iter=5000, multi_class='multinomial')),
    ])
    acc_vals.append(cross_val_score(pipe, X, y, cv=cv, scoring='accuracy').mean())

plt.figure(figsize=(7.4, 4.1))
plt.plot(list(n_vals), acc_vals, marker='o')
plt.xlabel('n_components')
plt.ylabel('Accuracy CV (media)')
plt.title('Desempeno de PCA + regresion logistica')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
